# ChurnGuard API — Live Tests
Testing the deployed Render API: `https://customer-churn-prediction-02k2.onrender.com`

Covers: happy-path predictions, boundary/edge-case inputs, error-body structure, and auth guards.

In [ ]:
import requests

BASE = "https://customer-churn-prediction-02k2.onrender.com"

passed, failed = [], []

def check(name, condition, detail=""):
    if condition:
        passed.append(name)
        print(f"  PASS  {name}")
    else:
        failed.append(name)
        print(f"  FAIL  {name}" + (f" — {detail}" if detail else ""))

def check_422(name, r, expect_field=None):
    """Assert HTTP 422, a non-empty detail list, and optionally that expect_field appears in the error loc."""
    ok = r.status_code == 422
    check(f"{name} — 422 status", ok, f"got {r.status_code}")
    if not ok:
        return
    detail = r.json().get("detail", [])
    check(f"{name} — detail is a list", isinstance(detail, list) and len(detail) >= 1,
          f"detail={detail!r}")
    if expect_field and detail:
        field_found = any(expect_field in str(e.get("loc", "")) for e in detail)
        check(f"{name} — '{expect_field}' named in error",
              field_found, f"locs={[e.get('loc') for e in detail]}")

BANK_PAYLOAD = {
    "name": "Jane Smith", "credit_score": 650, "geography": "France",
    "gender": "Female", "age": 42, "tenure": 5, "balance": 75000.0,
    "num_products": 2, "has_cr_card": 1, "is_active_member": 1, "estimated_salary": 85000.0,
}

TELCO_PAYLOAD = {
    "name": "John Rivera", "gender": "Male", "senior_citizen": 0,
    "partner": "No", "dependents": "No", "tenure": 12,
    "phone_service": "Yes", "multiple_lines": "No",
    "internet_service": "Fiber optic", "online_security": "No",
    "online_backup": "No", "device_protection": "No", "tech_support": "No",
    "streaming_tv": "Yes", "streaming_movies": "Yes",
    "contract": "Month-to-month", "paperless_billing": "Yes",
    "payment_method": "Electronic check",
    "monthly_charges": 89.85, "total_charges": 1078.20,
}

print("Setup complete.")

## Meta

In [ ]:
r = requests.get(f"{BASE}/health", timeout=60)
b = r.json()
check("GET /health — status 200",     r.status_code == 200)
check("GET /health — status ok",      b.get("status") == "ok")
check("GET /health — 7 bank models",  b.get("bank_models") == 7)
check("GET /health — 5 telco models", b.get("telco_models") == 5)
print()

r = requests.get(f"{BASE}/models", timeout=60)
b = r.json()
check("GET /models — status 200",     r.status_code == 200)
check("GET /models — 7 bank models",  len(b.get("bank", [])) == 7)
check("GET /models — 5 telco models", len(b.get("telco", [])) == 5)
print("\nBank models: ",  b.get("bank"))
print("Telco models:", b.get("telco"))

## Bank Churn — Happy Path

In [ ]:
r = requests.post(f"{BASE}/predict/bank", json=BANK_PAYLOAD, timeout=60)
b = r.json()
prob = b.get("churn_probability", -1)

check("POST /predict/bank — status 200",           r.status_code == 200)
check("POST /predict/bank — probability in [0,1]", 0.0 <= prob <= 1.0)
check("POST /predict/bank — risk_level valid",     b.get("risk_level") in {"low", "medium", "high"})
check("POST /predict/bank — 7 model scores",       len(b.get("model_scores", {})) == 7)
check("POST /predict/bank — scores in [0,1]",      all(0 <= v <= 1 for v in b.get("model_scores", {}).values()))
check("POST /predict/bank — dataset label",        b.get("dataset") == "bank")
print(f"\nChurn probability: {prob:.1%}  |  Risk: {b.get('risk_level')}")
print("Model scores:")
for model, score in b.get("model_scores", {}).items():
    print(f"  {model:<22} {score:.4f}")

## Bank Churn — Edge Cases

All invalid requests must return HTTP **422** with a `detail` list that names the offending field.

In [ ]:
# High-risk profile: old, inactive, zero balance, max products
high_risk = {**BANK_PAYLOAD, "age": 55, "balance": 0, "is_active_member": 0, "num_products": 4}
r = requests.post(f"{BASE}/predict/bank", json=high_risk, timeout=60)
prob = r.json().get("churn_probability", 0)
check("POST /predict/bank — high-risk profile > 0.4", prob > 0.4, f"got {prob:.3f}")

In [ ]:
# credit_score below minimum (300)
check_422("bank credit_score=100",
    requests.post(f"{BASE}/predict/bank", json={**BANK_PAYLOAD, "credit_score": 100}, timeout=30),
    expect_field="credit_score")

# credit_score above maximum (850)
check_422("bank credit_score=900",
    requests.post(f"{BASE}/predict/bank", json={**BANK_PAYLOAD, "credit_score": 900}, timeout=30),
    expect_field="credit_score")

# age below minimum (18)
check_422("bank age=17",
    requests.post(f"{BASE}/predict/bank", json={**BANK_PAYLOAD, "age": 17}, timeout=30),
    expect_field="age")

# age above maximum (100)
check_422("bank age=150",
    requests.post(f"{BASE}/predict/bank", json={**BANK_PAYLOAD, "age": 150}, timeout=30),
    expect_field="age")

# gender not in Literal["Male","Female"]
check_422("bank gender=Unknown",
    requests.post(f"{BASE}/predict/bank", json={**BANK_PAYLOAD, "gender": "Unknown"}, timeout=30),
    expect_field="gender")

# num_products below minimum (1)
check_422("bank num_products=0",
    requests.post(f"{BASE}/predict/bank", json={**BANK_PAYLOAD, "num_products": 0}, timeout=30),
    expect_field="num_products")

# num_products above maximum (4)
check_422("bank num_products=5",
    requests.post(f"{BASE}/predict/bank", json={**BANK_PAYLOAD, "num_products": 5}, timeout=30),
    expect_field="num_products")

# balance below minimum (0)
check_422("bank balance=-1",
    requests.post(f"{BASE}/predict/bank", json={**BANK_PAYLOAD, "balance": -1}, timeout=30),
    expect_field="balance")

# has_cr_card not in Literal[0,1]
check_422("bank has_cr_card=2",
    requests.post(f"{BASE}/predict/bank", json={**BANK_PAYLOAD, "has_cr_card": 2}, timeout=30),
    expect_field="has_cr_card")

# name violates min_length=1
check_422("bank name=empty-string",
    requests.post(f"{BASE}/predict/bank", json={**BANK_PAYLOAD, "name": ""}, timeout=30),
    expect_field="name")

## Telco Churn — Happy Path

In [ ]:
r = requests.post(f"{BASE}/predict/telco", json=TELCO_PAYLOAD, timeout=60)
b = r.json()
prob = b.get("churn_probability", -1)

check("POST /predict/telco — status 200",           r.status_code == 200)
check("POST /predict/telco — probability in [0,1]", 0.0 <= prob <= 1.0)
check("POST /predict/telco — risk_level valid",     b.get("risk_level") in {"low", "medium", "high"})
check("POST /predict/telco — 5 model scores",       len(b.get("model_scores", {})) == 5)
check("POST /predict/telco — scores in [0,1]",      all(0 <= v <= 1 for v in b.get("model_scores", {}).values()))
check("POST /predict/telco — dataset label",        b.get("dataset") == "telco")
print(f"\nChurn probability: {prob:.1%}  |  Risk: {b.get('risk_level')}")
print("Model scores:")
for model, score in b.get("model_scores", {}).items():
    print(f"  {model:<22} {score:.4f}")

## Telco Churn — Edge Cases

All invalid requests must return HTTP **422** with a `detail` list that names the offending field.

In [ ]:
# Low-risk profile: long contract, long tenure, low monthly charges
low_risk = {**TELCO_PAYLOAD, "contract": "Two year", "tenure": 60, "monthly_charges": 30.0}
r = requests.post(f"{BASE}/predict/telco", json=low_risk, timeout=60)
prob = r.json().get("churn_probability", 1)
check("POST /predict/telco — low-risk profile < 0.6", prob < 0.6, f"got {prob:.3f}")

In [ ]:
# contract not a recognised literal
check_422("telco contract=Weekly",
    requests.post(f"{BASE}/predict/telco", json={**TELCO_PAYLOAD, "contract": "Weekly"}, timeout=30),
    expect_field="contract")

# internet_service not a recognised literal
check_422("telco internet_service=Cable",
    requests.post(f"{BASE}/predict/telco", json={**TELCO_PAYLOAD, "internet_service": "Cable"}, timeout=30),
    expect_field="internet_service")

# payment_method not a recognised literal
check_422("telco payment_method=Bitcoin",
    requests.post(f"{BASE}/predict/telco", json={**TELCO_PAYLOAD, "payment_method": "Bitcoin"}, timeout=30),
    expect_field="payment_method")

# gender not in Literal["Male","Female"]
check_422("telco gender=NonBinary",
    requests.post(f"{BASE}/predict/telco", json={**TELCO_PAYLOAD, "gender": "NonBinary"}, timeout=30),
    expect_field="gender")

# senior_citizen not in Literal[0,1]
check_422("telco senior_citizen=2",
    requests.post(f"{BASE}/predict/telco", json={**TELCO_PAYLOAD, "senior_citizen": 2}, timeout=30),
    expect_field="senior_citizen")

# tenure below minimum (0)
check_422("telco tenure=-1",
    requests.post(f"{BASE}/predict/telco", json={**TELCO_PAYLOAD, "tenure": -1}, timeout=30),
    expect_field="tenure")

# monthly_charges below minimum (0)
check_422("telco monthly_charges=-10",
    requests.post(f"{BASE}/predict/telco", json={**TELCO_PAYLOAD, "monthly_charges": -10}, timeout=30),
    expect_field="monthly_charges")

## Train Endpoint — Auth Guard

The `/train/run` endpoint requires an `X-Train-Key` header.
- No header or wrong key → **403** (key configured) or **503** (key not configured on the server).
- Either way the response must contain a plain-English `detail` string.

In [ ]:
# No auth header — expect 4xx/5xx, never 200
r_no_key = requests.post(f"{BASE}/train/run", timeout=30)
check("POST /train/run — no key → not 200",
      r_no_key.status_code != 200,
      f"got {r_no_key.status_code}")
check("POST /train/run — no key → 4xx or 5xx",
      r_no_key.status_code in {403, 422, 503},
      f"got {r_no_key.status_code}")
check("POST /train/run — no key → detail in body",
      "detail" in r_no_key.json(),
      f"body={r_no_key.json()}")

# Wrong key — same expectations
r_bad_key = requests.post(f"{BASE}/train/run",
    headers={"X-Train-Key": "not-the-real-key"},
    timeout=30)
check("POST /train/run — wrong key → not 200",
      r_bad_key.status_code != 200,
      f"got {r_bad_key.status_code}")
check("POST /train/run — wrong key → 4xx or 5xx",
      r_bad_key.status_code in {403, 422, 503},
      f"got {r_bad_key.status_code}")
check("POST /train/run — wrong key → detail in body",
      "detail" in r_bad_key.json(),
      f"body={r_bad_key.json()}")

print(f"\nNo-key response  ({r_no_key.status_code}): {r_no_key.json().get('detail')}")
print(f"Wrong-key response ({r_bad_key.status_code}): {r_bad_key.json().get('detail')}")

## Summary

In [ ]:
total = len(passed) + len(failed)
print(f"Results: {len(passed)}/{total} passed")
if failed:
    print("\nFailed:")
    for name in failed:
        print(f"  FAIL  {name}")